# Preprocessing Notebook
Cleans df1 (diabetes health indicators), df2 (diabetic hospital data), and df3 (stroke data).
Outputs three ready-to-use CSVs for modeling notebooks.

In [1]:
import pandas as pd
import numpy as np

## Load Raw Data

In [2]:
df1 = pd.read_csv('diabetes_health_indicators.csv')
df2 = pd.read_csv('diabetic_data.csv')
df3 = pd.read_csv('stroke_data.csv')

print(f'df1 shape: {df1.shape}')
print(f'df2 shape: {df2.shape}')
print(f'df3 shape: {df3.shape}')

df1 shape: (433323, 20)
df2 shape: (101766, 50)
df3 shape: (5110, 12)


---
## DF1 — Diabetes Health Indicators
**Target:** `DIABETE4` (0 = not diabetic, 1 = diabetic)

In [3]:
df1_clean = df1.copy()

# --- Drop DIABTYPE: 94% missing, only populated for confirmed diabetics ---
df1_clean.drop(columns=['DIABTYPE'], inplace=True)

# --- Replace don't know / refused / missing sentinel values with NaN ---
df1_clean['DIABETE4'] = df1_clean['DIABETE4'].replace([7, 9], np.nan)
df1_clean['_RFHYPE6'] = df1_clean['_RFHYPE6'].replace([9], np.nan)
df1_clean['_RFCHOL3'] = df1_clean['_RFCHOL3'].replace([9], np.nan)
df1_clean['_CHOLCH3'] = df1_clean['_CHOLCH3'].replace([9], np.nan)
df1_clean['SMOKE100'] = df1_clean['SMOKE100'].replace([7, 9], np.nan)
df1_clean['CVDSTRK3'] = df1_clean['CVDSTRK3'].replace([7, 9], np.nan)
df1_clean['EXERANY2'] = df1_clean['EXERANY2'].replace([7, 9], np.nan)
df1_clean['PRIMINS1'] = df1_clean['PRIMINS1'].replace([77, 99], np.nan)
df1_clean['MEDCOST1'] = df1_clean['MEDCOST1'].replace([7, 9], np.nan)
df1_clean['GENHLTH']  = df1_clean['GENHLTH'].replace([7, 9], np.nan)
df1_clean['MENTHLTH'] = df1_clean['MENTHLTH'].replace([77, 99], np.nan)
df1_clean['PHYSHLTH'] = df1_clean['PHYSHLTH'].replace([77, 99], np.nan)
df1_clean['DIFFWALK'] = df1_clean['DIFFWALK'].replace([7, 9], np.nan)
df1_clean['_AGEG5YR'] = df1_clean['_AGEG5YR'].replace([14], np.nan)
df1_clean['_EDUCAG']  = df1_clean['_EDUCAG'].replace([9], np.nan)
df1_clean['_INCOMG1'] = df1_clean['_INCOMG1'].replace([9], np.nan)

# --- Remap sentinel 'none' (88) to 0 for day-count columns ---
df1_clean['MENTHLTH'] = df1_clean['MENTHLTH'].replace(88, 0)
df1_clean['PHYSHLTH'] = df1_clean['PHYSHLTH'].replace(88, 0)

# --- Recode Yes/No columns to binary: 1=Yes, 0=No ---
binary_cols = ['SMOKE100', 'CVDSTRK3', 'EXERANY2', 'MEDCOST1', 'DIFFWALK']
for col in binary_cols:
    df1_clean[col] = df1_clean[col].map({1: 1, 2: 0})

# _RFHYPE6 and _RFCHOL3 are flipped: 1=No, 2=Yes
df1_clean['_RFHYPE6'] = df1_clean['_RFHYPE6'].map({1: 0, 2: 1})
df1_clean['_RFCHOL3'] = df1_clean['_RFCHOL3'].map({1: 0, 2: 1})

# _MICHD: 1=had MI/CHD, 2=did not
df1_clean['_MICHD'] = df1_clean['_MICHD'].map({1: 1, 2: 0})

# _SEX: 0=Male, 1=Female
df1_clean['_SEX'] = df1_clean['_SEX'].map({1: 0, 2: 1})

# --- BMI: has 2 implied decimal places ---
df1_clean['_BMI5'] = df1_clean['_BMI5'] / 100

# --- DIABETE4: remap to clean binary target ---
# Original: 1=diabetic, 3=no, 2=gestational, 4=pre-diabetic
# Remapped: 1=diabetic, 0=not diabetic, 2=gestational, 3=pre-diabetic
df1_clean['DIABETE4'] = df1_clean['DIABETE4'].map({1: 1, 3: 0, 2: 2, 4: 3})

# --- Impute remaining NaNs with median ---
for col in df1_clean.columns:
    if df1_clean[col].isna().any():
        df1_clean[col] = df1_clean[col].fillna(df1_clean[col].median())

print('DF1 preprocessing complete')
print(f'Shape: {df1_clean.shape}')
print(f'\nMissing values:\n{df1_clean.isnull().sum()[df1_clean.isnull().sum() > 0]}')

DF1 preprocessing complete
Shape: (433323, 19)

Missing values:
Series([], dtype: int64)


---
## DF2 — Diabetic Hospital Data
**Target:** `readmitted` (0 = not readmitted within 30 days, 1 = readmitted within 30 days)

In [4]:
df2_clean = df2.copy()

# --- Replace '?' with NaN ---
df2_clean.replace('?', np.nan, inplace=True)

# --- Drop ID columns ---
df2_clean.drop(columns=['encounter_id', 'patient_nbr'], inplace=True)

# --- Drop near-constant columns ---
df2_clean.drop(columns=['examide', 'citoglipton'], inplace=True)

# --- Drop high-missingness + high-cardinality columns ---
df2_clean.drop(columns=['weight', 'payer_code', 'medical_specialty'], inplace=True)

# --- Age: extract midpoint from interval string ---
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
    '[80-90)': 85, '[90-100)': 95
}
df2_clean['age'] = df2_clean['age'].map(age_map)

# --- Gender: binary encode ---
df2_clean['gender'] = df2_clean['gender'].replace('Unknown/Invalid', np.nan)
df2_clean['gender'] = df2_clean['gender'].map({'Male': 0, 'Female': 1})

# --- Race: one-hot encode ---
df2_clean = pd.get_dummies(df2_clean, columns=['race'], dummy_na=True)

# --- ICD9 diagnosis codes: group into broad disease categories ---
def map_icd9(code):
    if pd.isna(code):
        return 'Unknown'
    code = str(code)
    if code.startswith('V') or code.startswith('E'):
        return 'Other'
    try:
        c = float(code)
        if 390 <= c <= 459 or c == 785: return 'Circulatory'
        elif 460 <= c <= 519 or c == 786: return 'Respiratory'
        elif 520 <= c <= 579 or c == 787: return 'Digestive'
        elif c == 250: return 'Diabetes'
        elif 800 <= c <= 999: return 'Injury'
        elif 710 <= c <= 739: return 'Musculoskeletal'
        elif 580 <= c <= 629 or c == 788: return 'Genitourinary'
        elif 140 <= c <= 239: return 'Neoplasms'
        else: return 'Other'
    except:
        return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df2_clean[col] = df2_clean[col].apply(map_icd9)
df2_clean = pd.get_dummies(df2_clean, columns=['diag_1', 'diag_2', 'diag_3'])

# --- Drug columns: ordinal encode (no=0, steady=1, up=2, down=3) ---
drug_map = {'No': 0, 'Steady': 1, 'Up': 2, 'Down': 3}
drug_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'insulin', 'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone'
]
for col in drug_cols:
    df2_clean[col] = df2_clean[col].map(drug_map)

# --- change and diabetesMed: binary encode ---
df2_clean['change'] = df2_clean['change'].map({'Ch': 1, 'No': 0})
df2_clean['diabetesMed'] = df2_clean['diabetesMed'].map({'Yes': 1, 'No': 0})

# --- max_glu_serum and A1Cresult: ordinal encode ---
df2_clean['max_glu_serum'] = df2_clean['max_glu_serum'].map({'None': 0, 'Norm': 1, '>200': 2, '>300': 3})
df2_clean['A1Cresult']     = df2_clean['A1Cresult'].map({'None': 0, 'Norm': 1, '>7': 2, '>8': 3})

# fill NaNs left from '?' replacement before the map ran
df2_clean['max_glu_serum'] = df2_clean['max_glu_serum'].fillna(0)
df2_clean['A1Cresult']     = df2_clean['A1Cresult'].fillna(0)
df2_clean['gender']        = df2_clean['gender'].fillna(df2_clean['gender'].mode()[0])

# --- admission_type_id, discharge_disposition_id, admission_source_id: one-hot encode ---
df2_clean = pd.get_dummies(df2_clean, columns=[
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id'
])

# --- Target: binary readmission (<30 days = 1, everything else = 0) ---
df2_clean['readmitted'] = df2_clean['readmitted'].map({'NO': 0, '>30': 0, '<30': 1})

print('DF2 preprocessing complete')
print(f'Shape: {df2_clean.shape}')
print(f'\nRemaining missing values:\n{df2_clean.isnull().sum()[df2_clean.isnull().sum() > 0]}')

DF2 preprocessing complete
Shape: (101766, 123)

Remaining missing values:
Series([], dtype: int64)


---
## DF3 — Stroke Data
**Target:** `stroke` (0 = no stroke, 1 = stroke)

In [5]:
df3_clean = df3.copy()

# --- Drop ID column ---
df3_clean.drop(columns=['id'], inplace=True)

# --- Gender: drop the single 'Other' row, binary encode ---
df3_clean['gender'] = df3_clean['gender'].replace('Other', np.nan)
df3_clean.dropna(subset=['gender'], inplace=True)
df3_clean['gender'] = df3_clean['gender'].map({'Male': 0, 'Female': 1})

# --- Ever married: binary encode ---
df3_clean['ever_married'] = df3_clean['ever_married'].map({'No': 0, 'Yes': 1})

# --- Residence type: binary encode ---
df3_clean['Residence_type'] = df3_clean['Residence_type'].map({'Rural': 0, 'Urban': 1})

# --- Work type: one-hot encode ---
df3_clean = pd.get_dummies(df3_clean, columns=['work_type'])

# --- Smoking status: keep 'Unknown' as its own category (informative missingness) ---
df3_clean = pd.get_dummies(df3_clean, columns=['smoking_status'])

# --- BMI: impute missing with median (~4% missing, safe to impute) ---
df3_clean['bmi'] = df3_clean['bmi'].fillna(df3_clean['bmi'].median())

# --- hypertension, heart_disease, stroke: already binary, nothing to do ---

print('DF3 preprocessing complete')
print(f'Shape: {df3_clean.shape}')
print(f'\nClass balance:')
print(df3_clean['stroke'].value_counts(normalize=True).round(3))
print(f'\nMissing values:\n{df3_clean.isnull().sum()[df3_clean.isnull().sum() > 0]}')

DF3 preprocessing complete
Shape: (5109, 18)

Class balance:
stroke
0    0.951
1    0.049
Name: proportion, dtype: float64

Missing values:
Series([], dtype: int64)


---
## Save Cleaned Datasets

In [6]:
df1_clean.to_csv('df1_preprocessed.csv', index=False)
df2_clean.to_csv('df2_preprocessed.csv', index=False)
df3_clean.to_csv('df3_preprocessed.csv', index=False)

print('Saved:')
print(f'  df1_preprocessed.csv  — {df1_clean.shape[0]:,} rows, {df1_clean.shape[1]} columns')
print(f'  df2_preprocessed.csv  — {df2_clean.shape[0]:,} rows, {df2_clean.shape[1]} columns')
print(f'  df3_preprocessed.csv  — {df3_clean.shape[0]:,} rows, {df3_clean.shape[1]} columns')
print('\nIn your modeling notebooks, load with:')
print("  df1 = pd.read_csv('df1_preprocessed.csv')")
print("  df2 = pd.read_csv('df2_preprocessed.csv')")
print("  df3 = pd.read_csv('df3_preprocessed.csv')")

Saved:
  df1_preprocessed.csv  — 433,323 rows, 19 columns
  df2_preprocessed.csv  — 101,766 rows, 123 columns
  df3_preprocessed.csv  — 5,109 rows, 18 columns

In your modeling notebooks, load with:
  df1 = pd.read_csv('df1_preprocessed.csv')
  df2 = pd.read_csv('df2_preprocessed.csv')
  df3 = pd.read_csv('df3_preprocessed.csv')
